In [1]:
import scanpy as sc
import pandas as pd
import os
import mygene
mg = mygene.MyGeneInfo()

In [2]:
metadata = pd.read_csv('/Users/christoffer/work/karolinska/development/data/abc_atlas/metadata/WMB-10X/20241115/cell_metadata.csv')
cluster_to_anno = pd.read_csv('/Users/christoffer/work/karolinska/development/Allen_ABC/data/abc_atlas/metadata/WMB-taxonomy/20231215/views/cluster_to_cluster_annotation_membership_pivoted.csv')
base_dir = '/Users/christoffer/work/karolinska/development/data/abc_atlas/expression_matrices/WMB-10Xv2/20230630/'
files = os.listdir(base_dir)

In [3]:
files = [
    'WMB-10Xv2-Isocortex-1-raw.h5ad',
    'WMB-10Xv2-OLF-raw.h5ad',
    'WMB-10Xv2-CTXsp-raw.h5ad',
    'WMB-10Xv2-TH-raw.h5ad',
    #'WMB-10Xv2-Isocortex-2-raw.h5ad',
    #'WMB-10Xv2-Isocortex-3-raw.h5ad',
    'WMB-10Xv2-MB-raw.h5ad',
    'WMB-10Xv2-HY-raw.h5ad',
    'WMB-10Xv2-HPF-raw.h5ad'
]

In [4]:
ad_list = []
for file in files: 
    print(file)
    ad_ = sc.read_h5ad(base_dir+file)
    ad_list.append(ad_)

WMB-10Xv2-Isocortex-1-raw.h5ad
WMB-10Xv2-OLF-raw.h5ad
WMB-10Xv2-CTXsp-raw.h5ad
WMB-10Xv2-TH-raw.h5ad
WMB-10Xv2-MB-raw.h5ad
WMB-10Xv2-HY-raw.h5ad
WMB-10Xv2-HPF-raw.h5ad


In [5]:
ad = sc.concat(
    ad_list,
)

In [6]:
ad.obs

,cell_barcode,library_label,anatomical_division_label
cell_label,,,
ATTACTCCAAGTAATG-010_A01,ATTACTCCAAGTAATG,L8TX_180221_01_B11,Isocortex
CATCAAGTCTGTTTGT-099_E01,CATCAAGTCTGTTTGT,L8TX_190312_01_A03,Isocortex
CGATCGGAGCTCTCGG-099_A01,CGATCGGAGCTCTCGG,L8TX_190312_01_E02,Isocortex
CGTGTAACAAGAAGAG-099_A01,CGTGTAACAAGAAGAG,L8TX_190312_01_E02,Isocortex
TGTATTCAGCGATATA-099_C01,TGTATTCAGCGATATA,L8TX_190312_01_F02,Isocortex
...,...,...,...
TGACAACGTTCACGGC-014_F01,TGACAACGTTCACGGC,L8TX_180221_01_G12,HPF
GCGAGAAAGTCATCCA-024_A01,GCGAGAAAGTCATCCA,L8TX_180406_01_D06,HPF
GACACGCAGACAGACC-024_B01,GACACGCAGACAGACC,L8TX_180406_01_F06,HPF


In [7]:
del ad_list

In [8]:
for meta in ['cluster_alias', 'donor_sex', 'dataset_label','x','y']:
    mapping_dict = dict(zip(metadata['cell_barcode'], metadata[meta]))
    ad.obs[meta] = ad.obs['cell_barcode'].map(mapping_dict)

In [9]:
for meta in ['neurotransmitter', 'class', 'subclass', 'supertype','cluster']:
    mapping_dict = dict(zip(cluster_to_anno['cluster_alias'], cluster_to_anno[meta]))
    ad.obs[meta] = ad.obs['cluster_alias'].map(mapping_dict)

In [10]:
ad.obs['class'].value_counts()

class
01 IT-ET Glut        391820
31 OPC-Oligo         123190
02 NP-CT-L6b Glut     79513
06 CTX-CGE GABA       70285
05 OB-IMN GABA        68688
18 TH Glut            66584
07 CTX-MGE GABA       62776
04 DG-IMN Glut        22227
30 Astro-Epen         19464
34 Immune              8774
33 Vascular            7618
12 HY GABA             7034
14 HY Glut             6087
13 CNU-HYa Glut        4227
19 MB Glut             4224
20 MB GABA             4003
16 HY MM Glut          2694
09 CNU-LGE GABA        2436
08 CNU-MGE GABA        1988
11 CNU-HYa GABA        1419
03 OB-CR Glut          1382
17 MH-LH Glut           783
32 OEC                  109
25 Pineal Glut           89
10 LSX GABA              37
24 MY Glut               19
21 MB Dopa               15
23 P Glut                13
27 MY GABA               12
26 P GABA                11
15 HY Gnrh1 Glut          4
28 CB GABA                3
Name: count, dtype: int64

In [13]:
# Normalizing to median total counts
sc.pp.normalize_total(ad)
# Logarithmize the data
sc.pp.log1p(ad)

In [14]:
ad.write('/Users/christoffer/work/karolinska/development/data/abc_atlas/combined_scRNAseq.h5ad')

In [27]:
ad_sp = sc.read_h5ad('../data/mtDNA_DSB_5k_clustered_LLM_anno.h5ad')


/Users/christoffer/miniconda3/envs/sc/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


""
gene_identifier
ENSMUSG00000051951
ENSMUSG00000089699
ENSMUSG00000102331
ENSMUSG00000102343
ENSMUSG00000025900
...
ENSMUSG00000095523
ENSMUSG00000095475
ENSMUSG00000094855


In [12]:
import re
import pandas as pd
from pathlib import Path

def load_ensembl_gene_map(gtf_path):
    """
    Parse an Ensembl GTF and return a dict: {ensembl_gene_id: gene_symbol}.
    Keeps only 'gene' features; strips version suffixes.
    """
    gene_map = {}
    pat_id   = re.compile(r'gene_id "([^"]+)"')
    pat_name = re.compile(r'gene_name "([^"]+)"')
    with open(gtf_path, "r") as fh:
        for line in fh:
            if line.startswith("#"): 
                continue
            # Only keep 'gene' feature lines to avoid huge memory use
            # GTF columns: seqname, source, feature, start, end, score, strand, frame, attributes
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 9 or parts[2] != "gene":
                continue
            attrs = parts[8]
            m_id = pat_id.search(attrs)
            m_nm = pat_name.search(attrs)
            if not (m_id and m_nm):
                continue
            gid = m_id.group(1).split('.')[0]   # strip version, e.g. ENSMUSG... .1 → base
            gnm = m_nm.group(1)
            # keep first occurrence (usually fine). If you want, prefer protein_coding using parts[1]/attrs
            if gid not in gene_map:
                gene_map[gid] = gnm
    return gene_map

In [13]:


# --- use it ---
# Point to your Mus musculus Ensembl GTF (e.g., Mus_musculus.GRCm39.109.gtf)
gtf_file = "//Users/christoffer/Downloads/Mus_musculus.GRCm39.109.gtf"
gene_map = load_ensembl_gene_map(gtf_file)

# ids_to_map: list/Series/Index of 32,285 Ensembl IDs (with/without version)
def map_ids_to_symbols(ids, gene_map):
    # vectorized mapping via pandas for speed and NaN handling
    s = pd.Series(ids, dtype="string")
    base = s.str.replace(r"\.\d+$", "", regex=True)  # strip version suffixes
    symbols = base.map(gene_map).astype("string")
    return symbols.tolist()

# Example:
# ids_to_map = adata.var_names.tolist()
# symbols = map_ids_to_symbols(ids_to_map, gene_map)
# 

In [15]:
ids_to_map = ad.var_names.tolist()

In [16]:
symbols = map_ids_to_symbols(ids_to_map, gene_map)

In [19]:
ad.var["gene_symbol"] = symbols

In [25]:
ad.var = ad.var.set_index('gene_symbol')

In [28]:
# intersect gene sets
common_genes = ad.var_names.intersection(ad_sp.var_names)

# filter both to the same set of genes (if you want to align them)
ad_sub = ad[:, common_genes].copy()

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [29]:
common_genes

Index(['Sox17', 'Rgs20', 'Oprk1', 'Npbwr1', 'Rb1cc1', 'Vxn', 'Cops5', 'Sulf1',
       'Prdm14', 'Ncoa2',
       ...
       'Ace2', 'Vegfd', 'Asb9', 'Glra2', 'Ofd1', 'Tlr8', 'Tlr7', 'Amelx',
       'Uba1y', 'Sry'],
      dtype='object', length=5054)

In [ ]:
ad.var_names_make_unique()     # appends -1, -2… to duplicates
ad_sub = ad[:, common_genes].copy()

In [30]:
ad[:, common_genes]

InvalidIndexError: Reindexing only valid with uniquely valued Index objects